# NB11 — Packaging and Publishing (pyproject.toml, TestPyPI)

**Module 16 — Research Software Engineering**

---

## Motivation

A library that only works when you clone the repository from a specific path is not a tool — it's a pile of files. Packaging makes `compbio_utils` installable with a single `pip install` command, from anywhere, by anyone. Publishing to PyPI (or TestPyPI) makes it citable, versioned, and discoverable. This is the difference between code that exists and software that is used.

## Intuition

`pyproject.toml` is the modern single file that contains everything about your package: its name, version, author, dependencies, development tools, and build system. Running `python -m build` reads this file and produces two distributable artifacts: a `.tar.gz` source distribution (sdist) and a `.whl` wheel (binary distribution).

## Background: What is Semantic Versioning?

MAJOR.MINOR.PATCH:
- **MAJOR**: incompatible API change (rename a function, change a return type)
- **MINOR**: backward-compatible new functionality (add a new function)
- **PATCH**: backward-compatible bug fix

0.1.0 means: first public release (0.x.y means "not yet stable").

---

In [ ]:
import textwrap
import tempfile
from pathlib import Path

# The complete pyproject.toml for compbio_utils
PYPROJECT_TOML = textwrap.dedent("""
    [build-system]
    requires = ["hatchling"]
    build-backend = "hatchling.build"

    [project]
    name = "compbio-utils"
    version = "0.1.0"
    description = "Shared computational biology utilities for sequence analysis, statistics, and file I/O."
    readme = "README.md"
    license = {file = "LICENSE"}
    authors = [
        {name = "Vinoth", email = "vinoth.ac.in@gmail.com"},
    ]
    keywords = ["bioinformatics", "computational biology", "sequence analysis"]
    classifiers = [
        "Development Status :: 3 - Alpha",
        "Intended Audience :: Science/Research",
        "License :: OSI Approved :: MIT License",
        "Programming Language :: Python :: 3.10",
        "Programming Language :: Python :: 3.11",
        "Programming Language :: Python :: 3.12",
        "Topic :: Scientific/Engineering :: Bio-Informatics",
    ]
    requires-python = ">=3.10"
    dependencies = [
        "numpy>=1.24",
        "pandas>=2.0",
    ]

    [project.optional-dependencies]
    dev = [
        "pytest>=7.0",
        "pytest-cov>=4.0",
        "ruff>=0.4",
        "mypy>=1.0",
        "click>=8.0",
    ]
    docs = [
        "sphinx>=7.0",
        "sphinx-rtd-theme>=2.0",
    ]

    [project.scripts]
    compbio = "compbio_utils.cli:cli"

    [project.urls]
    Homepage = "https://github.com/Vinoth-ai-20/computational-biology"
    Repository = "https://github.com/Vinoth-ai-20/computational-biology"
    "Bug Tracker" = "https://github.com/Vinoth-ai-20/computational-biology/issues"

    [tool.hatch.build.targets.wheel]
    packages = ["src/compbio_utils"]

    [tool.pytest.ini_options]
    testpaths = ["tests"]
    addopts = ["-v", "--tb=short"]

    [tool.ruff]
    line-length = 88
    target-version = "py310"

    [tool.ruff.lint]
    select = ["E", "F", "I", "B", "UP", "ANN"]
    ignore = ["ANN101", "ANN102"]

    [tool.mypy]
    strict = true
    python_version = "3.10"
""")

print('Generated pyproject.toml:')
print(PYPROJECT_TOML)

In [ ]:
# Write to a temp directory and validate structure
import tomllib  # Python 3.11+ stdlib; use tomli for 3.10

tmpdir = Path(tempfile.mkdtemp(prefix='pyproject_demo_'))
pyproject_path = tmpdir / 'pyproject.toml'
pyproject_path.write_text(PYPROJECT_TOML)

# Parse and validate
with pyproject_path.open('rb') as f:
    config = tomllib.load(f)

project = config['project']
print('Validation checks:')
checks = [
    ('name',            'name' in project),
    ('version',         'version' in project),
    ('description',     'description' in project),
    ('python >=3.10',   project.get('requires-python', '') >= '>=3.10'),
    ('has numpy dep',   any('numpy' in d for d in project.get('dependencies', []))),
    ('has pandas dep',  any('pandas' in d for d in project.get('dependencies', []))),
    ('has CLI entry',   'compbio' in config.get('project', {}).get('scripts', {})),
    ('has dev extras',  'dev' in config.get('project', {}).get('optional-dependencies', {})),
    ('has ruff config', 'ruff' in config.get('tool', {})),
    ('has mypy config', 'mypy' in config.get('tool', {})),
]

all_pass = True
for name, result in checks:
    status = 'PASS' if result else 'FAIL'
    print(f'  [{status}] {name}')
    all_pass = all_pass and result

print(f'\nOverall: {"ALL PASS" if all_pass else "SOME CHECKS FAILED"}')

In [ ]:
# Publishing workflow (commands — not run here, they require TestPyPI account)
publish_commands = textwrap.dedent("""
    # Step 1: Build the package
    cd utilities/compbio_utils/
    python -m build
    # → Creates dist/compbio_utils-0.1.0.tar.gz  (sdist)
    #   and  dist/compbio_utils-0.1.0-py3-none-any.whl (wheel)

    # Step 2: Check the package
    twine check dist/*
    # → Should output: PASSED for both files

    # Step 3: Upload to TestPyPI (not real PyPI)
    twine upload --repository testpypi dist/*
    # → Prompts for TestPyPI API token
    # → URL: https://test.pypi.org/project/compbio-utils/

    # Step 4: Test the install in a fresh virtual environment
    python -m venv /tmp/test_env
    source /tmp/test_env/bin/activate
    pip install --index-url https://test.pypi.org/simple/ compbio-utils
    python -c 'from compbio_utils import gc_content; print(gc_content("ATCG"))'
    # → 0.5

    # Step 5: Tag the release
    git tag v0.1.0
    git push origin v0.1.0
    # → GitHub can create a Release from this tag
    # → Zenodo watches the repo and mints a DOI automatically
""")

print('Publishing workflow (run manually):')
print(publish_commands)

## Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('Python Packaging: pyproject.toml and Publishing', fontsize=13, fontweight='bold')

# --- Panel 1: Packaging workflow diagram ---
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 12)
ax1.axis('off')
ax1.set_title('Packaging Workflow', fontweight='bold')

steps = [
    (5, 11.0, 'Write code\n(src/compbio_utils/*.py)', '#2c3e50'),
    (5,  9.0, 'pyproject.toml\n[project] + [build-system]', '#3498db'),
    (5,  7.0, 'python -m build\n→ dist/*.tar.gz + *.whl', '#9b59b6'),
    (5,  5.0, 'twine check dist/*\n→ Validate metadata', '#e67e22'),
    (5,  3.0, 'twine upload --repository testpypi\n→ Test install', '#27ae60'),
    (5,  1.0, 'git tag v0.1.0 + Zenodo DOI\n→ Citable software', '#e74c3c'),
]

for x, y, label, color in steps:
    bbox = mpatches.FancyBboxPatch((x - 3.5, y - 0.6), 7, 1.2,
                                    boxstyle='round,pad=0.1',
                                    facecolor=color, alpha=0.85, edgecolor='white')
    ax1.add_patch(bbox)
    ax1.text(x, y, label, ha='center', va='center', fontsize=8.5,
             color='white', fontweight='bold')

for i in range(len(steps) - 1):
    _, y1, _, _ = steps[i]
    _, y2, _, _ = steps[i + 1]
    ax1.annotate('', xy=(5, y2 + 0.6), xytext=(5, y1 - 0.6),
                 arrowprops=dict(arrowstyle='->', color='#555', lw=2))

# --- Panel 2: Semantic versioning scheme ---
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('Semantic Versioning', fontweight='bold')

# Version number boxes
ax2.text(5, 9, '0  .  1  .  0', ha='center', fontsize=28,
         fontfamily='monospace', fontweight='bold', color='#2c3e50')

labels = [
    (1.5, 7.5, 'MAJOR', 'Incompatible\nAPI change', '#e74c3c'),
    (5.0, 7.5, 'MINOR', 'New backward-\ncompatible feature', '#e67e22'),
    (8.5, 7.5, 'PATCH', 'Bug fix', '#27ae60'),
]

for x, y, name, desc, color in labels:
    ax2.text(x, y, name, ha='center', fontsize=11, fontweight='bold', color=color)
    ax2.text(x, y - 0.9, desc, ha='center', fontsize=8.5, color='#555')

# Version progression example
versions = [
    ('0.1.0', 'Initial release', '#27ae60'),
    ('0.1.1', 'Fixed bug in gc_content', '#27ae60'),
    ('0.2.0', 'Added translate() function', '#e67e22'),
    ('1.0.0', 'Stable public API', '#e74c3c'),
]

for i, (ver, msg, color) in enumerate(versions):
    y_pos = 5.5 - i * 1.1
    ax2.text(0.5, y_pos, ver, fontsize=11, fontfamily='monospace',
             fontweight='bold', color=color, va='center')
    ax2.text(2.2, y_pos, msg, fontsize=9, color='#555', va='center')
    if i > 0:
        ax2.annotate('', xy=(1.2, y_pos + 0.9), xytext=(1.2, y_pos + 0.15),
                     arrowprops=dict(arrowstyle='->', color='#aaa', lw=1.5))

plt.tight_layout()
plt.savefig('packaging_publishing.png', dpi=120, bbox_inches='tight')
plt.show()

## Exercises

See `exercises/README.md` → E11.

1. Write a minimal `pyproject.toml` for a package called `seqtools` with one dependency (`numpy>=1.24`) and one CLI entry point.
2. What is the difference between a wheel and a source distribution? When would you install from source?
3. What is an "editable install" (`pip install -e .`)? When is it useful during development?

## Quiz

1. What are the three sections in MAJOR.MINOR.PATCH and what triggers each?
2. What does `[project.scripts]` do in `pyproject.toml`?
3. What is the difference between `dependencies` and `optional-dependencies`?
4. What does `python -m build` produce?
5. What is TestPyPI used for?

## Reflection

*After completing:* Why is `0.1.0` (not `1.0.0`) the right version for a first release? What commitment does bumping to `1.0.0` imply?

## References

- Python Packages book: https://py-pkgs.org/
- Semantic versioning: https://semver.org/
- TestPyPI: https://test.pypi.org/